In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import torch
import accelerate
import pandas as pd
import os
import shutil
import numpy as np
from sklearn.model_selection import train_test_split

print(f"Accelerate 版本: {accelerate.__version__}")
print(f"PyTorch 版本: {torch.__version__}")

# 定义8种情绪映射
emotion_mapping = {
    "开心": 0, "难过": 1, "愤怒": 2, "恐惧": 3, 
    "惊讶": 4, "平静": 5, "厌恶": 6, "困惑": 7,
}

id_to_emotion = {v: k for k, v in emotion_mapping.items()}

print("🎭 情绪类别映射:")
for emotion, idx in emotion_mapping.items():
    print(f"  {emotion} -> {idx}")

# 使用正确的模型路径
model_path = "model/hfl_chinese-roberta-wwm-ext/models--hfl--chinese-roberta-wwm-ext/snapshots/5c58d0b8ec1d9014354d691c538661bf00bfdb44"
print(f"📁 使用模型路径: {model_path}")

# 首先检查CSV文件结构
print("检查CSV文件结构...")
df = pd.read_csv("text.csv")
print(f"数据行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")
print("\n标签分布:")
label_counts = df['label'].value_counts()
print(label_counts)
print("\n数据样例:")
print(df.head())

# 检查数据平衡性
print(f"\n📊 数据平衡性分析:")
min_count = label_counts.min()
max_count = label_counts.max()
imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
print(f"  最少样本的情绪: {label_counts.idxmin()} ({min_count}条)")
print(f"  最多样本的情绪: {label_counts.idxmax()} ({max_count}条)")
print(f"  不平衡比例: {imbalance_ratio:.2f}x")

if imbalance_ratio > 3:
    print("⚠️  数据不平衡，建议进行数据增强或重采样")

# 检查情绪标签是否都在定义的情绪中
unique_labels = df['label'].unique()
print(f"\n发现的情绪标签: {unique_labels}")
for label in unique_labels:
    if label not in emotion_mapping:
        print(f"⚠️  警告: 标签 '{label}' 不在预定义的情绪映射中")

# 将文本标签转换为数字ID
df['label_id'] = df['label'].map(emotion_mapping)

# 检查是否有未映射的标签
if df['label_id'].isna().any():
    print("❌ 错误: 存在未映射的情绪标签，请检查数据")
    unmapped = df[df['label_id'].isna()]
    print("未映射的数据:")
    print(unmapped[['text', 'label']])
    # 删除未映射的行
    df = df.dropna(subset=['label_id'])
    print(f"删除未映射行后数据量: {len(df)}")

# 分割训练集和验证集
print("\n📁 分割数据集...")
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

print(f"训练集: {len(train_df)} 条")
print(f"验证集: {len(eval_df)} 条")

# 保存处理后的数据到临时文件
train_csv = "temp_train_data.csv"
eval_csv = "temp_eval_data.csv"
train_df[['text', 'label_id']].to_csv(train_csv, index=False)
eval_df[['text', 'label_id']].to_csv(eval_csv, index=False)

print(f"保存训练数据到: {train_csv}")
print(f"保存验证数据到: {eval_csv}")

# 加载模型和分词器 - 使用正确的路径
print("\n正在从本地加载模型和分词器...")
try:
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        local_files_only=True,
        num_labels=len(emotion_mapping),
        id2label=id_to_emotion,
        label2id=emotion_mapping
    )
    print("✅ 模型和分词器从本地加载完成！")
except Exception as e:
    print(f"❌ 从本地加载失败: {e}")
    print("请检查模型文件是否完整")
    exit()

# 加载数据集
print("正在加载数据集...")
dataset = load_dataset("csv", data_files={
    "train": train_csv,
    "eval": eval_csv
})

# 数据预处理函数
def preprocess_function(examples):
    # 分词处理
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors=None
    )
    
    # 确保标签是整数类型
    tokenized["labels"] = [int(label) for label in examples["label_id"]]
    
    return tokenized

# 对数据集进行编码
print("预处理数据...")
encoded_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(f"训练集样本数: {len(encoded_dataset['train'])}")
print(f"验证集样本数: {len(encoded_dataset['eval'])}")

# 修正后的训练参数 - 兼容新老版本
training_args = TrainingArguments(
    output_dir="./temp_training_output",
    
    # 训练参数
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,  # 进一步增加训练轮次
    learning_rate=1.5e-5,  # 微调学习率
    
    # 优化参数
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    # 评估和保存 - 兼容新老版本
    eval_strategy="epoch",  # 新版本参数
    save_strategy="epoch",  # 新版本参数
    # evaluation_strategy="epoch",  # 老版本参数（已弃用）
    # save_strategy="epoch",       # 老版本参数（已弃用）
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    
    # 日志
    logging_steps=20,
    logging_dir="./temp_logs",
    report_to=None,
    
    # 其他优化
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    
    # 学习率调度
    lr_scheduler_type="linear",
)

# 如果上面的参数还是报错，使用这个最简版本：
# training_args = TrainingArguments(
#     output_dir="./temp_training_output",
#     per_device_train_batch_size=16,
#     num_train_epochs=20,
#     learning_rate=1.5e-5,
#     warmup_ratio=0.1,
#     weight_decay=0.01,
#     logging_steps=20,
#     save_steps=500,
#     save_total_limit=2,
#     report_to=None,
#     remove_unused_columns=False,
# )

# 创建训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["eval"],  # 添加验证集
    tokenizer=tokenizer,
)

print("\n🚀 开始优化训练多情绪分类模型...")
print(f"训练轮次: {training_args.num_train_epochs}")
print(f"批量大小: {training_args.per_device_train_batch_size}")
print(f"学习率: {training_args.learning_rate}")
print(f"情绪类别数: {len(emotion_mapping)}")
print(f"训练集大小: {len(encoded_dataset['train'])}")
print(f"验证集大小: {len(encoded_dataset['eval'])}")

# 训练模型
print("\n开始训练...")
train_results = trainer.train()

# 保存最佳模型
final_model_dir = "my_final_model/my_model_2"
print(f"\n💾 保存最佳模型到: {final_model_dir}")

# 确保目录存在
os.makedirs(final_model_dir, exist_ok=True)

# 保存模型和分词器
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("✅ 最佳模型保存完成！")

# 最终评估
print("\n📊 最终模型评估...")
eval_results = trainer.evaluate()
print(f"验证集损失: {eval_results['eval_loss']:.4f}")

# 清理临时文件
print("\n🧹 清理临时文件...")
for temp_file in [train_csv, eval_csv]:
    if os.path.exists(temp_file):
        os.remove(temp_file)
        print(f"删除临时文件: {temp_file}")

if os.path.exists("./temp_training_output"):
    shutil.rmtree("./temp_training_output")
    print("删除临时训练输出目录")

if os.path.exists("./temp_logs"):
    shutil.rmtree("./temp_logs")
    print("删除临时日志目录")

print(f"\n🎉 训练完成！")
print(f"💾 最佳模型保存在: {final_model_dir}")
print(f"📈 最终验证损失: {eval_results['eval_loss']:.4f}")

Accelerate 版本: 1.10.1
PyTorch 版本: 2.8.0+cu129
🎭 情绪类别映射:
  开心 -> 0
  难过 -> 1
  愤怒 -> 2
  恐惧 -> 3
  惊讶 -> 4
  平静 -> 5
  厌恶 -> 6
  困惑 -> 7
📁 使用模型路径: model/hfl_chinese-roberta-wwm-ext/models--hfl--chinese-roberta-wwm-ext/snapshots/5c58d0b8ec1d9014354d691c538661bf00bfdb44
检查CSV文件结构...
数据行数: 827
列名: ['text', 'label']

标签分布:
label
困惑    111
惊讶    107
平静    106
难过    104
愤怒    104
开心    102
恐惧     99
厌恶     94
Name: count, dtype: int64

数据样例:
                text label
0  听到这个好消息，我开心得跳了起来！    开心
1  终于完成了这个项目，太有成就感了！    开心
2    和朋友相聚的时光总是这么愉快！    开心
3   看到花开得这么美，心情都变好了！    开心
4   吃到想念已久的美食，幸福感爆棚！    开心

📊 数据平衡性分析:
  最少样本的情绪: 厌恶 (94条)
  最多样本的情绪: 困惑 (111条)
  不平衡比例: 1.18x

发现的情绪标签: ['开心' '难过' '愤怒' '恐惧' '惊讶' '平静' '厌恶' '困惑']

📁 分割数据集...
训练集: 661 条
验证集: 166 条
保存训练数据到: temp_train_data.csv
保存验证数据到: temp_eval_data.csv

正在从本地加载模型和分词器...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at model/hfl_chinese-roberta-wwm-ext/models--hfl--chinese-roberta-wwm-ext/snapshots/5c58d0b8ec1d9014354d691c538661bf00bfdb44 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ 模型和分词器从本地加载完成！
正在加载数据集...


Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

...

C:\Users\gsh\AppData\Local\Temp\ipykernel_24748\1146101233.py:194: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.



🚀 开始优化训练多情绪分类模型...
训练轮次: 20
批量大小: 16
学习率: 1.5e-05
情绪类别数: 8
训练集大小: 661
验证集大小: 166

开始训练...


Epoch,Training Loss,Validation Loss
1,1.950000,1.773511
2,1.143300,0.620224
3,0.325000,0.126623
4,0.089400,0.035467
5,0.028300,0.031840
6,0.023300,0.042201
7,0.012000,0.012593
8,0.010800,0.022198
9,0.021600,0.016312
10,0.018600,0.014954



💾 保存最佳模型到: my_final_model/my_model_2
✅ 最佳模型保存完成！

📊 最终模型评估...


验证集损失: 0.0126

🧹 清理临时文件...
删除临时文件: temp_train_data.csv
删除临时文件: temp_eval_data.csv
删除临时训练输出目录

🎉 训练完成！
💾 最佳模型保存在: my_final_model/my_model_2
📈 最终验证损失: 0.0126


In [3]:
# 手动保存最终模型到指定目录
final_model_dir = "my_final_model/my_model_2"
print(f"\n💾 保存最终模型到: {final_model_dir}")

# 确保目录存在
os.makedirs(final_model_dir, exist_ok=True)

# 保存模型和分词器
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("✅ 最终模型保存完成！")

# 清理临时文件
print("\n🧹 清理临时文件...")
if os.path.exists(temp_csv):
    os.remove(temp_csv)
    print(f"删除临时文件: {temp_csv}")

if os.path.exists("./temp_training_output"):
    shutil.rmtree("./temp_training_output")
    print("删除临时训练输出目录")

if os.path.exists("./temp_logs"):
    shutil.rmtree("./temp_logs")
    print("删除临时日志目录")

print(f"\n🎉 训练完成！")
print(f"💾 最终模型保存在: {final_model_dir}")
print(f"📊 支持的情绪: {list(emotion_mapping.keys())}")


💾 保存最终模型到: my_final_model/my_model_2
✅ 最终模型保存完成！

🧹 清理临时文件...
删除临时文件: temp_emotion_data.csv
删除临时训练输出目录

🎉 训练完成！
💾 最终模型保存在: my_final_model/my_model_2
📊 支持的情绪: ['开心', '难过', '愤怒', '恐惧', '惊讶', '平静', '厌恶', '困惑']


In [1]:
# diagnose_model_path.py
import os

def diagnose_model():
    print("🔍 诊断模型文件位置...")
    
    # 检查你指定的路径
    specified_path = "model/hfl_chinese-roberta-wwm-ext"
    print(f"1. 检查指定路径: {specified_path}")
    print(f"   路径存在: {os.path.exists(specified_path)}")
    
    if os.path.exists(specified_path):
        print(f"   是否是目录: {os.path.isdir(specified_path)}")
        if os.path.isdir(specified_path):
            files = os.listdir(specified_path)
            print(f"   目录内容: {files}")
    
    # 检查当前目录结构
    print(f"\n2. 当前目录结构:")
    for item in os.listdir('.'):
        if os.path.isdir(item):
            print(f"   📁 {item}/")
        else:
            print(f"   📄 {item}")
    
    # 搜索模型文件
    print(f"\n3. 搜索模型文件...")
    model_files_found = []
    
    for root, dirs, files in os.walk('.'):
        for file in files:
            if file in ['config.json', 'pytorch_model.bin', 'vocab.txt', 'tokenizer.json']:
                full_path = os.path.join(root, file)
                model_files_found.append(full_path)
    
    if model_files_found:
        print("   找到的模型文件:")
        for file in model_files_found:
            print(f"   - {file}")
    else:
        print("   ❌ 没有找到模型文件")
    
    # 检查可能的模型目录
    print(f"\n4. 可能的模型目录:")
    possible_dirs = [
        "model",
        "hfl_chinese-roberta-wwm-ext", 
        "chinese-roberta-wwm-ext",
        "roberta",
        "saved_model"
    ]
    
    for dir_name in possible_dirs:
        if os.path.exists(dir_name) and os.path.isdir(dir_name):
            print(f"   📁 {dir_name}/")
            files = os.listdir(dir_name)
            for file in files:
                print(f"      - {file}")

if __name__ == "__main__":
    diagnose_model()

🔍 诊断模型文件位置...
1. 检查指定路径: model/hfl_chinese-roberta-wwm-ext
   路径存在: True
   是否是目录: True
   目录内容: ['.locks', 'models--hfl--chinese-roberta-wwm-ext']

2. 当前目录结构:
   📁 .ipynb_checkpoints/
   📁 .virtual_documents/
   📄 1_main.ipynb
   📄 1_test.ipynb
   📄 2_main.ipynb
   📄 2_test.ipynb
   📁 anaconda_projects/
   📁 model/
   📁 my_final_model/
   📁 output/
   📄 temp_emotion_data.csv
   📁 temp_training/
   📁 temp_training_output/
   📄 text.csv

3. 搜索模型文件...
   找到的模型文件:
   - .\model\hfl_chinese-roberta-wwm-ext\models--hfl--chinese-roberta-wwm-ext\snapshots\5c58d0b8ec1d9014354d691c538661bf00bfdb44\config.json
   - .\model\hfl_chinese-roberta-wwm-ext\models--hfl--chinese-roberta-wwm-ext\snapshots\5c58d0b8ec1d9014354d691c538661bf00bfdb44\pytorch_model.bin
   - .\model\hfl_chinese-roberta-wwm-ext\models--hfl--chinese-roberta-wwm-ext\snapshots\5c58d0b8ec1d9014354d691c538661bf00bfdb44\tokenizer.json
   - .\model\hfl_chinese-roberta-wwm-ext\models--hfl--chinese-roberta-wwm-ext\snapshots\5c58d0b8ec1d90